In [1]:
import os
import shutil
from pyspark import SparkConf, SparkContext

# Spark Configuration Setup
conf = SparkConf().setAppName("EmployeeRDDJupyter").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 10:06:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
def parse_csv(line):
    parts = line.split(',')
    return int(parts[0]), parts[1], parts[2], float(parts[3])

# RDD loading and filtering header
raw_rdd = sc.textFile("employees.csv")
header = raw_rdd.first()
data_rdd = raw_rdd.filter(lambda line: line != header).map(parse_csv)

print("Data loaded into RDD successfully.")

[Stage 0:>                                                          (0 + 1) / 1]

Data loaded into RDD successfully.


In [6]:
sorted_rdd = data_rdd.map(lambda x: (x[3], (x[0], x[1], x[2]))).sortByKey(ascending=False)

print("Employees sorted by salary (Descending):")
for salary, details in sorted_rdd.collect():
    print(f"ID: {details[0]} | Name: {details[1]} | Dept: {details[2]} | Salary: {salary}")

Employees sorted by salary (Descending):
ID: 4 | Name: Priya | Dept: Finance | Salary: 70000.0
ID: 3 | Name: Neha | Dept: IT | Salary: 65000.0
ID: 7 | Name: Rohit | Dept: Finance | Salary: 60000.0
ID: 1 | Name: Amit | Dept: IT | Salary: 55000.0
ID: 5 | Name: Karan | Dept: IT | Salary: 50000.0
ID: 6 | Name: Simran | Dept: HR | Salary: 45000.0
ID: 2 | Name: Rahul | Dept: HR | Salary: 40000.0


In [7]:
dept_salary_rdd = data_rdd.map(lambda x: (x[2], x[3])).reduceByKey(lambda a, b: a + b)

print("Department-wise Total Salaries:")
for dept, total_salary in dept_salary_rdd.collect():
    print(f"Department: {dept:<10} | Total Salary: {total_salary}")

Department-wise Total Salaries:
Department: IT         | Total Salary: 170000.0
Department: HR         | Total Salary: 85000.0
Department: Finance    | Total Salary: 130000.0


In [8]:
top_3_employees = sorted_rdd.take(3)

output_lines = ["id,name,department,salary"]
for salary, details in top_3_employees:
    output_lines.append(f"{details[0]},{details[1]},{details[2]},{salary}")

output_dir = "top_3_output"
# Agar purana folder exist karta hai toh use delete karein takis Spark error na de
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

sc.parallelize(output_lines).coalesce(1).saveAsTextFile(output_dir)
print(f"Saved top 3 employees folder to '{output_dir}/'")

Saved top 3 employees folder to 'top_3_output/'
